# Wikontic tutorial

This notebook walks through:

1. **Triplet extraction** with an LLM (no database)
2. **Dynamic KG construction** — entity/relation refinement without Wikidata ontology
3. **Structured KG construction** — Wikidata-aligned types, properties, and constraints
4. **Question answering** over the stored graph
5. **LangChain agents** wrapping Wikontic tools

Paper: [arXiv:2512.00590](https://arxiv.org/abs/2512.00590)

## Setup

Run from the repository root (clone the repo first):

```bash
pip install -r requirements.txt
pip install -e .
pip install langchain langchain-openai   # optional, for the agent section
```

Wikontic uses the `facebook/contriever` embedding model. Download it once for offline use:

```bash
pip install huggingface_hub
huggingface-cli download facebook/contriever --local-dir models/facebook--contriever
```

Or set `HF_MODEL_PATH` to an existing local checkout.

In [ ]:
import os

import httpx
from dotenv import load_dotenv, find_dotenv

from wikontic.utils.openai_utils import LLMTripletExtractor
from wikontic.utils.dynamic_aligner import Aligner as DynamicAligner
from wikontic.utils.structured_aligner import Aligner as StructuredAligner
from wikontic.utils.inference_with_db import InferenceWithDB
from wikontic.utils.structured_inference_with_db import StructuredInferenceWithDB
from wikontic.create_triplets_db import create_triplets_database
from wikontic.create_ontological_triplets_db import create_ontological_triplets_database
from wikontic.create_wikidata_ontology_db import create_wikidata_ontology_database

## Before we start

Create a `.env` file in the **repository root** (not necessarily next to this notebook):

| Variable | Purpose |
|----------|----------|
| `OPENROUTER_BASE_URL` | OpenAI-compatible API base URL |
| `OPENROUTER_KEY` or `KEY` | API key |
| `PROXY_URL` | Optional HTTP proxy |
| `MONGO_URI` | MongoDB URI (default: `mongodb://localhost:27018/?directConnection=true`) |
| `HF_MODEL_PATH` | Optional local path to `facebook/contriever` weights |

In [ ]:
_ = load_dotenv(find_dotenv())

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27018/?directConnection=true")
base_url = os.getenv("OPENROUTER_BASE_URL")
api_key = os.getenv("OPENROUTER_KEY") or os.getenv("KEY")
proxy_url = os.getenv("PROXY_URL")

http_client = httpx.Client(proxy=proxy_url) if proxy_url else httpx.Client()
DEVICE = "cpu"  # use "cuda" if a GPU is available

## Extract triplets (LLM only)

Using `LLMTripletExtractor`, you can extract factual triplets `(subject, relation, object)`.

The structured extraction prompt also returns:

- `subject_type`, `object_type` — coarse entity classes
- `qualifiers` — list of `{relation, object}` dicts attached to the main triplet

In [ ]:
extractor = LLMTripletExtractor(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
    proxy=proxy_url,
)

text_1 = """
Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor,
and investor best known for co-founding the technology company Apple Inc.
"""

triplets = extractor.extract_triplets_from_text(text=text_1)
triplets

## Database setup

Wikontic stores triplets and embedding indexes in a **storage backend** (`mongodb` or `qdrant`).

For MongoDB Atlas Local:

```bash
docker pull mongodb/mongodb-atlas-local:latest
docker run -d --name wikontic -p 27018:27018 mongodb/mongodb-atlas-local:latest
```

The `create_*_database` helpers return a backend object with `upsert_many`, `vector_search`, and `match_documents`.

## Dynamic KG pipeline (no ontology)

Use this when you do **not** need Wikidata type/property constraints — only embedding-based entity and relation linking.

In [ ]:
triplets_db = create_triplets_database(
    backend="mongodb",
    mongo_uri=MONGO_URI,
    db_name="tutorial_non_onto",
    drop_collections=True,
)

dynamic_aligner = DynamicAligner(triplets_db=triplets_db, device=DEVICE)
dynamic_inferer = InferenceWithDB(extractor, dynamic_aligner, triplets_db)

initial, final, filtered = dynamic_inferer.extract_triplets_and_add_to_db(
    text=text_1,
    sample_id="wikipedia",
    source_text_id="1",
)
print(f"initial={len(initial)}, final={len(final)}, filtered={len(filtered)}")
final

### QA over the dynamic KG

In [ ]:
linked_entities = dynamic_inferer.identify_relevant_entities_from_question_with_llm(
    "Who was Jobs?",
    use_entity_types=False,
)
supporting_triplets, answer = dynamic_inferer.answer_question_with_llm(
    "Who was Jobs?",
    linked_entities,
)
answer

### Dynamic pipeline as LangChain tools

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url=base_url,
    api_key=api_key,
    http_client=http_client,
)

dynamic_agent = create_agent(
    llm,
    tools=[
        dynamic_inferer.extract_triplets_and_add_to_db_tool,
        dynamic_inferer.retrieve_similar_entity_names_tool,
        dynamic_inferer.get_1_hop_supporting_triplets_tool,
        dynamic_inferer.answer_question_with_llm_tool,
    ],
)

text_2 = """Jobs was also the founder of NeXT and chairman and majority shareholder of Pixar.
He was a pioneer of the personal computer revolution of the 1970s and 1980s,
along with his early business partner and fellow Apple co-founder Steve Wozniak."""

dynamic_agent.invoke({"messages": [{"role": "user", "content": text_2}]})

In [ ]:
dynamic_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What are the companies Jobs founded?"}
        ]
    }
)

---

## Structured KG pipeline (Wikidata ontology)

Structured mode refines entity types and relations against a Wikidata-derived ontology, then validates triplets against property constraints.

Triplet collections written to the database:

| Collection | Description |
|------------|-------------|
| `initial_triplets` | Raw LLM output |
| `triplets` | Accepted triplets after refinement |
| `filtered_triplets` | Triplets dropped due to errors during refinement |
| `ontology_filtered_triplets` | Triplets that violate ontology constraints |

In [ ]:
ontology_db = create_wikidata_ontology_database(
    backend="mongodb",
    mongo_uri=MONGO_URI,
    database="tutorial_wikidata_ontology",
    drop_collections=True,
)

onto_triplets_db = create_ontological_triplets_database(
    backend="mongodb",
    mongo_uri=MONGO_URI,
    db_name="tutorial_onto",
    drop_collections=True,
)

structured_aligner = StructuredAligner(
    ontology_db=ontology_db,
    triplets_db=onto_triplets_db,
    device=DEVICE,
)
structured_inferer = StructuredInferenceWithDB(
    extractor, structured_aligner, onto_triplets_db
)

In [ ]:
initial, final, filtered, ontology_filtered = (
    structured_inferer.extract_triplets_with_ontology_filtering_and_add_to_db(
        text=text_1,
        sample_id="wikipedia",
        source_text_id="1",
    )
)
print(
    f"initial={len(initial)}, final={len(final)}, "
    f"filtered={len(filtered)}, ontology_filtered={len(ontology_filtered)}"
)
final

### QA over the structured KG

In [ ]:
linked_entities = structured_inferer.identify_relevant_entities_from_question_with_llm(
    "When was the person that founded Apple born?"
)
supporting_triplets, answer = structured_inferer.answer_question_with_llm(
    "When was the person that founded Apple born?",
    linked_entities=linked_entities,
)
answer

### Structured pipeline as LangChain tools

In [ ]:
structured_agent = create_agent(
    llm,
    tools=[
        structured_inferer.extract_triplets_with_ontology_filtering_and_add_to_db_tool,
        structured_inferer.retrieve_similar_entity_names_tool,
        structured_inferer.get_1_hop_supporting_triplets_tool,
        structured_inferer.answer_question_with_llm_tool,
    ],
)

structured_agent.invoke({"messages": [{"role": "user", "content": text_2}]})

In [ ]:
structured_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What are the companies Jobs founded?"}
        ]
    }
)

---

## Optional: Qdrant in-memory backend

For ephemeral runs (tests, batch jobs without MongoDB), set `vector_db_backend: qdrant` and `qdrant_url: :memory:` in a config file.

`dataset_inference.py` builds the ontology and triplets DB in one process and automatically writes `kg_dump/kg_dump_{db_name}.json` when using in-memory Qdrant.

```bash
python inference_and_eval/dataset_inference.py \
  --config inference_and_eval/configs/musique_inference_qdrant.yaml
```

To export a KG from any backend in Python:

```python
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "analysis"))
from dump_kg import dump_kg_from_backend

# `backend` is the StorageBackend returned by create_*_database
dump_path = dump_kg_from_backend(backend, "my_run", include_ontology_filtered=True)
```